In [2]:
import re                               # for document class
import xml.etree.ElementTree as ET      # for parsing xml
import os
import pandas as pd
# text processor
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
# end text processor
# encoplot engine
import subprocess
from sentence_transformers import SentenceTransformer, util
import torch
# end encoplot engine
import shutil                             # copy dataset
# for dbscan
from sklearn.cluster import DBSCAN
import numpy as np

In [ ]:
from google.colab import drive

# mount the drive
drive.mount('/content/drive')

# define the paths for the dataset
BASE_PATH = "/content/drive/MyDrive/PAN11/external-detection-corpus"

SOURCE_PATH = os.path.join(BASE_PATH, "source-document")
SUSPICIOUS_PATH = os.path.join(BASE_PATH, "suspicious-document")

# define the paths for the encoplot code
ENCOPLOT_PATH = "/content/drive/MyDrive/utils/encoplot.c"
EXECUTABLE_PATH = "./encoplot_engine"

print(f"Lookin for data in: {BASE_PATH}")

In [4]:
!gcc -O3 {ENCOPLOT_PATH} -o {EXECUTABLE_PATH}

In [5]:
%%writefile mass_encoplot.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <omp.h>

#define halfword_t unsigned long long
typedef struct { halfword_t lo, hi; } word_t;

#define eqword(x,y) (x.lo==y.lo && x.hi==y.hi)
#define ltword(x,y) (x.hi<y.hi || (x.hi==y.hi && x.lo<y.lo))
#define readat(x,y,z) {x.lo=*(halfword_t *)(y+z); x.hi=*(halfword_t *)(y+z+8);}

typedef struct {
    unsigned char *buf;
    int *ix;
    int len;
} processed_doc;

void simpler_rsort(unsigned char *x, int l, int DEPTH, int *retval) {
    int NN = l - DEPTH + 1;
    if (NN <= 0) return;
    int *ox = (int*)malloc(NN * sizeof(int));
    int counters[256], startpos[256];
    for (int i = 0; i < NN; i++) retval[i] = i;
    for (int j = 0; j < DEPTH; j++) {
        memset(counters, 0, sizeof(counters));
        for (int i = 0; i < NN; i++) counters[x[j + retval[i]]]++;
        int sp = 0;
        for (int k = 0; k < 256; k++) { startpos[k] = sp; sp += counters[k]; }
        for (int i = 0; i < NN; i++) {
            unsigned char c = x[j + retval[i]];
            ox[startpos[c]++] = retval[i];
        }
        memcpy(retval, ox, NN * sizeof(int));
    }
    free(ox);
}

processed_doc process_file(const char *path) {
    processed_doc doc = {NULL, NULL, 0};
    FILE *f = fopen(path, "rb");
    if (!f) return doc;
    fseek(f, 0, SEEK_END);
    doc.len = ftell(f);
    rewind(f);
    doc.buf = (unsigned char *)malloc(doc.len + 20);
    fread(doc.buf, 1, doc.len, f);
    fclose(f);
    doc.ix = (int *)malloc(doc.len * sizeof(int));
    simpler_rsort(doc.buf, doc.len, 16, doc.ix);
    return doc;
}

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "Utilizare: %s suspicios.txt sursa1.txt sursa2.txt ...\n", argv[0]);
        return 1;
    }

    processed_doc susp = process_file(argv[1]);
    if (!susp.buf) return 1;

    int depth = 16;
    int susp_limit = susp.len - (depth - 1);

    #pragma omp parallel for schedule(dynamic)
    for (int i = 2; i < argc; i++) {
        processed_doc src = process_file(argv[i]);
        if (!src.buf) continue;

        int src_limit = src.len - (depth - 1);
        int i1 = 0, i2 = 0;

        while (i1 < susp_limit && i2 < src_limit) {
            word_t s1; readat(s1, susp.buf, susp.ix[i1]);
            word_t s2; readat(s2, src.buf, src.ix[i2]);

            if (eqword(s1, s2)) {
                #pragma omp critical
                printf("%d %d %d\n", i, susp.ix[i1], src.ix[i2]);
                i1++; i2++;
            } else if (ltword(s1, s2)) { i1++; } else { i2++; }
        }
        free(src.buf); free(src.ix);
    }

    free(susp.buf); free(susp.ix);
    return 0;
}

Writing mass_encoplot.c


In [6]:
!gcc -O3 -fopenmp mass_encoplot.c -o mass_encoplot

mass_encoplot.c: In function ‘process_file’:
mass_encoplot.c:47:5: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   47 |     fread(doc.buf, 1, doc.len, f);
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~


## Segment

In [7]:
class Segment:
  def __init__(self, raw_text, start_index, length, clean_text=None):
    self.raw_text = raw_text
    self.start_index = start_index
    self.length = length
    self.clean_text = clean_text

    #self.embedding_vec = None
    #self.pos_tags = None
    #self.concept_ids = None

  def __eq__(self, other):
    s_start, s_end = self.start_index, self.start_index + self.length
    o_start, o_end = other.start_index, other.start_index + other.length

    MAX_DIF = 50
    return not (s_end + MAX_DIF < o_start or o_end + MAX_DIF < s_start)

  def __repr__(self):
    return "offset: " + str(self.start_index) + " length: " + str(self.length)

In [8]:
class PlagiarismFeature:
    def __init__(self, this_offset, this_length, source_reference,
                 source_offset, source_length, obfuscation):
        self.this_offset = this_offset
        self.this_length = this_length
        self.source_reference = source_reference
        self.source_offset = source_offset
        self.source_length = source_length
        self.obfuscation = obfuscation

    def get_source_id(self):
        return int(self.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))

    def __repr__(self):
        return (f"PlagiarismFeature("
                f"offset={self.this_offset}, "
                f"length={self.this_length}, "
                f"source={self.source_reference}, "
                f"obfuscation={self.obfuscation})")

In [9]:
class Document:
  def __init__(self, doc_id, is_source=True):
    self.numeric_id = int(doc_id)
    self.is_source = is_source

    formatted_id = f"{self.numeric_id:05d}"
    prefix = "source-document" if is_source else "suspicious-document"

    part_number = ((self.numeric_id - 1) // 500) + 1
    part_folder = f"part{part_number}"
    base_folder = SOURCE_PATH if is_source else SUSPICIOUS_PATH

    self.doc_name = f"{prefix}{formatted_id}.txt"
    self.xml_name = f"{prefix}{formatted_id}.xml"

    self.file_path = os.path.join(base_folder, part_folder, self.doc_name)
    self.xml_path = os.path.join(base_folder, part_folder, self.xml_name)

    with open(self.file_path, "r", encoding='utf-8', errors='ignore') as f:
      self.text = f.read()

    self.metadata = self._parse_xml()
    self.language = self.metadata.get('lang', 'english')
    self.segments = []

  def _parse_xml(self):
    meta = {}
    self.plagiarism_features = []
    try:
        tree = ET.parse(self.xml_path)
        root = tree.getroot()

        for feature in root.findall('feature'):
            if feature.get('name') == 'about':
                meta['lang'] = feature.get('lang', 'en')

            if feature.get('name') == 'md5Hash':
                meta['md5'] = feature.get('value')

            if feature.get('name') == 'plagiarism':
                self.plagiarism_features.append(PlagiarismFeature(
                    this_offset=int(feature.get('this_offset')),
                    this_length=int(feature.get('this_length')),
                    source_reference=feature.get('source_reference'),
                    source_offset=int(feature.get('source_offset')),
                    source_length=int(feature.get('source_length')),
                    obfuscation=feature.get('obfuscation', 'none')
                ))
    except Exception as e:
        print(f"Error parsing XML: {e}")
    return meta

  def extract_segments(self, anchor_offsets, window_size=256):
    self.segments = []
    for offset in anchor_offsets:
      start = max(0, offset)

      if start >= len(self.text): continue

      while start > 0 and self.text[start-1] not in [' ', '\n', '\t']:
        start -= 1

      end = min(len(self.text), start + window_size)

      while end < len(self.text) and self.text[end] not in [' ', '\n', '.', '!', '?']:
        end += 1

      self.segments.append(Segment(self.text[start:end], start, end - start))

In [10]:
class EncoplotEngine:
  def __init__(self, executable_path="./encoplot_engine"):
    self.executable_path = executable_path

  def get_anchors(self, path_susp, path_src):
    result = subprocess.run([self.executable_path, path_susp, path_src],
                            capture_output=True, text=True)
    if result.returncode != 0: return []

    lines = result.stdout.strip().split('\n')

    return [list(map(int, line.split())) for line in lines if line]

In [11]:
class SemanticAnalyzer:
  def __init__(self, model_name='paraphrase-multilingual-mpnet-base-v2'):  # paraphrase-multilingual-MiniLM-L12-v2
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.model = SentenceTransformer(model_name).to(self.device)

  def analyze_pairs(self, doc_susp, doc_src, threshold=0.6):
    texts_susp = [re.sub(r'\s+', ' ', s.raw_text).strip().lower() for s in doc_susp.segments]
    texts_src = [re.sub(r'\s+', ' ', s.raw_text).strip().lower() for s in doc_src.segments]

    embeddings_susp = self.model.encode(texts_susp, convert_to_tensor=True, show_progress_bar=True)
    embeddings_src = self.model.encode(texts_src, convert_to_tensor=True, show_progress_bar=True)

    cosine_scores = torch.nn.functional.cosine_similarity(embeddings_susp, embeddings_src)

    confirmed = []
    for i, score in enumerate(cosine_scores):
      if score >= threshold:
        confirmed.append({
            'score': score.item(),
            'susp_offset': doc_susp.segments[i].start_index,
            'susp_len': doc_susp.segments[i].length,
            'src_offset': doc_src.segments[i].start_index,
            'src_len': doc_src.segments[i].length,
            'text': texts_susp[i]
        })

    return confirmed

In [12]:
class EncoplotResult:
    def __init__(self, susp_id, src_id, anchors):
        self.susp_id = susp_id
        self.src_id = src_id
        self.anchors = anchors # list of [off_susp, off_src]
        self.anchor_count = len(anchors)
        # density score
        self.density_score = self._calculate_density()

    def _calculate_density(self):
        if self.anchor_count < 2: return 0
        offsets = [a[0] for a in self.anchors]
        span = max(offsets) - min(offsets)
        return self.anchor_count / (span / 1000) if span > 0 else 0

In [72]:
def get_safe_fragment(text, offset, win=450):
        if not text:
            return "", 0, 0

        text_len = len(text)
        start = max(0, min(offset, text_len - 1))

        while start > 0:
            if text[start-1] in [' ', '\n', '\t']:
                break
            start -= 1

        end = min(text_len, start + win)

        while end < len(text) and text[end] not in [' ', '\n', '.', '!', '?']:
            end += 1

        fragment = text[start:end]
        return fragment, start, end - start

!pip install --force-reinstall --no-cache-dir nltk

- daca nu merge nltk

In [14]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [15]:
class TextPreprocessor:
  def __init__(self, language='english'):
     self.lemmatizer = WordNetLemmatizer()
     try:
      self.stop_words = set(stopwords.words(language))
     except:
      self.stop_words = set(stopwords.words('english'))

  def process_document(self, doc):
    for seg in doc.segments:
      text = seg.raw_text.lower()
      text = re.sub(r"'s\b|'s", "", text)
      text = re.sub(r'[^a-z\s]', ' ', text)
      text = re.sub(r'\s+', ' ', text).strip()

      tokens = nltk.word_tokenize(text)

      cleaned_tokens = [
          self.lemmatizer.lemmatize(w)
          for w in tokens
          if w not in self.stop_words and len(w) > 2
          ]

      seg.clean_text = " ".join(cleaned_tokens)

      seg.pos_tags = nltk.pos_tag(nltk.word_tokenize(seg.raw_text))

    print(f"[OK] final preprocessing for {len(doc.segments)} segments.")

In [16]:
class ValidationMetrics:
    def __init__(self):
        self.recalls = []
        self.precisions = []
        self.granularities = []

    def calculate_metrics(self, ground_truth_features, predicted_segments):
        if not ground_truth_features:
            return {"precision": 0, "recall": 1, "granularity": 1, "f1": 0}

        # recall
        total_recall = 0
        for gt in ground_truth_features:
            covered_len = 0
            for pred in predicted_segments:
                intersect_start = max(gt.this_offset, pred.susp_off)
                intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)

                if intersect_end > intersect_start:
                    covered_len += (intersect_end - intersect_start)

            total_recall += (covered_len / gt.this_length)

        avg_recall = total_recall / len(ground_truth_features)

        # precision
        total_precision = 0
        if not predicted_segments:
            avg_precision = 0
        else:
            for pred in predicted_segments:
                covered_len = 0
                for gt in ground_truth_features:
                    intersect_start = max(gt.this_offset, pred.susp_off)
                    intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                    if intersect_end > intersect_start:
                        covered_len += (intersect_end - intersect_start)
                total_precision += (covered_len / pred.susp_len)
            avg_precision = total_precision / len(predicted_segments)

        # granularity
        avg_gran = 1.0

        # F-measure (plagdet score)
        f1 = 0
        if avg_precision + avg_recall > 0:
            f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)

        return {
            "recall": round(avg_recall, 4),
            "precision": round(avg_precision, 4),
            "f1": round(f1, 4)
        }

In [17]:
def extract_sample(n_suspicious):
    suspicious_docs = []
    source_ids_needed = set()

    for i in range(1, n_suspicious + 1):
        try:
            doc = Document(i, is_source=False)
            suspicious_docs.append(doc)

            for pf in doc.plagiarism_features:
                src_id = int(pf.source_reference
                               .replace("source-document", "")
                               .replace(".txt", ""))
                source_ids_needed.add(src_id)

        except Exception as e:
            print(f"[!] Suspicious {i:05d} error: {e}")
            continue

    print(f"Loaded suspicious docs : {len(suspicious_docs)}")
    print(f"Unique sources : {len(source_ids_needed)}")

    return suspicious_docs, sorted(source_ids_needed)

In [18]:
from sentence_transformers import SentenceTransformer, util

class SemanticComparator:
    def __init__(self, model_name='paraphrase-multilingual-mpnet-base-v2'):
        print("Loading SBERT model...")
        self.model = SentenceTransformer(model_name)

    def calculate_similarity(self, segment_susp, segment_src):
        embedding1 = self.model.encode(segment_susp.clean_text, convert_to_tensor=True)
        embedding2 = self.model.encode(segment_src.clean_text, convert_to_tensor=True)

        cosine_score = util.cos_sim(embedding1, embedding2)

        return cosine_score.item()

In [ ]:
suspicious_docs, source_ids = extract_sample(100)

KeyboardInterrupt: 

In [19]:
encoplot = EncoplotEngine()
analyzer = SemanticAnalyzer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def prefilter_with_tfidf(texts_susp, texts_src, all_meta, tfidf_threshold=0.05):
    print(f"Prefilltering with tf-idf on {len(texts_susp)} pairs")

    all_texts = texts_susp + texts_src
    vectorizer = TfidfVectorizer(max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    n = len(texts_susp)
    susp_matrix = tfidf_matrix[:n]
    src_matrix  = tfidf_matrix[n:]

    filtered_susp, filtered_src, filtered_meta = [], [], []

    BATCH = 10000
    for i in range(0, n, BATCH):
        batch_susp = susp_matrix[i:i+BATCH]
        batch_src  = src_matrix[i:i+BATCH]

        scores = np.array(batch_susp.multiply(batch_src).sum(axis=1)).flatten()

        for j, score in enumerate(scores):
            if score >= tfidf_threshold:
                filtered_susp.append(texts_susp[i+j])
                filtered_src.append(texts_src[i+j])
                filtered_meta.append(all_meta[i+j])

    print(f"  after tf-idf: {len(filtered_susp)} remaining pairs "
          f"(crossed out: {n - len(filtered_susp)})")
    return filtered_susp, filtered_src, filtered_meta

In [22]:
import time
import subprocess

class StandardEncoplotEngine:
    def __init__(self, executable_path="./encoplot_engine"):
        self.executable = executable_path

    def get_anchors(self, path_susp, path_src):
        result = subprocess.run([self.executable, path_susp, path_src], capture_output=True, text=True)
        return [list(map(int, line.split())) for line in result.stdout.strip().split('\n') if line]

class ParallelEncoplotEngine:
    def __init__(self, executable_path="./parallel_encoplot"):
        self.executable = executable_path

    def get_anchors(self, path_susp, path_src):
        result = subprocess.run([self.executable, path_susp, path_src], capture_output=True, text=True)
        return [list(map(int, line.split())) for line in result.stdout.strip().split('\n') if line]

In [23]:
susp_id = 175
source_ids = [3930, 7541, 10192, 7545]

In [25]:
import os
from concurrent.futures import ThreadPoolExecutor

def get_path(doc_id, is_source=True):
    prefix = "source" if is_source else "suspicious"
    part_num = ((int(doc_id) - 1) // 500) + 1
    file_name = f"{prefix}-document{int(doc_id):05d}.txt"
    return os.path.join(BASE_PATH, f"{prefix}-document", f"part{part_num}", file_name)

def run_mass_scan(susp_path, source_paths):
    cmd = ["./mass_encoplot", susp_path] + source_paths
    result = subprocess.run(cmd, capture_output=True, text=True)

    anchors_by_source = {path: [] for path in source_paths}

    for line in result.stdout.strip().split('\n'):
        if not line: continue
        parts = list(map(int, line.split()))
        if len(parts) == 3:
            s_idx, susp_off, src_off = parts
            source_path = source_paths[s_idx - 2]
            anchors_by_source[source_path].append([susp_off, src_off])

    return anchors_by_source

susp_id = 175
source_ids = list(range(700, 710))

susp_path = get_path(susp_id, is_source=False)
source_paths = [get_path(s_id, is_source=True) for s_id in source_ids if os.path.exists(get_path(s_id, is_source=True))]

print(f"Starting mass scan for {len(source_paths)} source documents...")
all_anchors = run_mass_scan(susp_path, source_paths)

for path, anchors in all_anchors.items():
    if anchors:
        print(f"Source {os.path.basename(path)} has {len(anchors)} anchors.")

Starting mass scan for 10 source documents...
Source source-document00700.txt has 153 anchors.
Source source-document00701.txt has 834 anchors.
Source source-document00702.txt has 69 anchors.
Source source-document00703.txt has 102 anchors.
Source source-document00704.txt has 718 anchors.
Source source-document00705.txt has 1170 anchors.
Source source-document00706.txt has 201 anchors.
Source source-document00707.txt has 61 anchors.
Source source-document00708.txt has 25 anchors.
Source source-document00709.txt has 92 anchors.


In [69]:
import time
import os

def benchmark_mass_comparison(susp_id, source_ids):
    susp_path = get_path(susp_id, is_source=False)
    source_paths = [get_path(s_id, is_source=True) for s_id in source_ids if os.path.exists(get_path(s_id, is_source=True))]

    print(f"BENCHMARK: 1 sus doc vs {len(source_paths)} sources")
    print("-" * 50)

    # standard encoplot
    engine_std = StandardEncoplotEngine("./encoplot_engine")
    start_old = time.time()
    anchors_old = {}
    for s_path in source_paths:
        anchors_old[s_path] = engine_std.get_anchors(susp_path, s_path)
    end_old = time.time()
    time_old = end_old - start_old

    print(f"Standard Encoplot: {time_old:.4f} sec")

    start_new = time.time()
    anchors_new = run_mass_scan(susp_path, source_paths)
    end_new = time.time()
    time_new = end_new - start_new

    print(f"Mass Scan:   {time_new:.4f} sec")

    speedup = time_old / time_new if time_new > 0 else 0
    print("-" * 50)
    print(f"Mass Scan is  x{speedup:.2f} times faster.")

    total_anchors_old = sum(len(v) for v in anchors_old.values())
    total_anchors_new = sum(len(v) for v in anchors_new.values())
    print(f"Anchors integrity: {'OK' if total_anchors_old == total_anchors_new else 'ERROR'}")

In [27]:
class PredictedSegment:
    def __init__(self, susp_id, src_id, susp_off, susp_len, src_off, src_len, score):
        self.susp_id = susp_id
        self.src_id = src_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_off = src_off
        self.src_len = src_len
        self.score = score

    def __repr__(self):
        return f"<Match Susp:{self.susp_id} Src:{self.src_id} Score:{self.score:.2f}>"

In [65]:
def cluster_results_dbscan(hits, eps=6000, min_samples=2, confidence_threshold=0.75):
    if not hits: return []

    X = np.array([[h.susp_off, h.src_off] for h in hits])

    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    labels = clustering.labels_

    merged_results = []
    unique_labels = set(labels)

    for label in unique_labels:
        if label == -1:
            continue

        cluster_hits = [hits[i] for i in range(len(hits)) if labels[i] == label]

        first_h = cluster_hits[0]

        min_susp_off = min(h.susp_off for h in cluster_hits)
        max_susp_end = max(h.susp_off + h.susp_len for h in cluster_hits)

        min_src_off = min(h.src_off for h in cluster_hits)
        max_src_end = max(h.src_off + h.src_len for h in cluster_hits)

        avg_score = sum(h.score for h in cluster_hits) / len(cluster_hits)

        merged_results.append(PredictedSegment(
            first_h.susp_id,
            first_h.src_id,
            min_susp_off,
            max_susp_end - min_susp_off,
            min_src_off,
            max_src_end - min_src_off,
            avg_score
        ))

    noise_indices = [i for i, label in enumerate(labels) if label == -1]

    for idx in noise_indices:
        hit = hits[idx]
        if hit.score >= confidence_threshold:
            merged_results.append(PredictedSegment(
                hit.susp_id,
                hit.src_id,
                hit.susp_off,
                hit.susp_len,
                hit.src_off,
                hit.src_len,
                hit.score
            ))

    return merged_results

In [1]:
import time
import re
import os
from collections import defaultdict
import xml.etree.ElementTree as ET
import torch

def run_pipeline(n_suspicious, analyzer, eps=6000, threshold=0.60):
    print("-" * 80)
    print(f"[STEP 1] Extracting sample: {n_suspicious} suspicious files")
    print("-" * 80)
    suspicious_docs, source_ids = extract_sample(n_suspicious)
    print(f"Necessary sources: {len(source_ids)}\n")

    print("[STEP 2] Loading source docs...")
    source_docs = []
    for s_id in source_ids:
        try:
            source_docs.append(Document(s_id, is_source=True))
        except Exception as e:
            print(f"  [!] Error loading source {s_id}: {e}")
    print(f"Sources loaded: {len(source_docs)}\n")

    print("[STEP 3] Encoplot - lexical radar...")
    t3 = time.time()

    all_texts_susp = []
    all_texts_src = []
    all_meta_pre_tfidf = []

    for doc_susp in suspicious_docs:
        src_paths = [d.file_path for d in source_docs]
        raw_results = run_mass_scan(doc_susp.file_path, src_paths)

        for s_path, anchors in raw_results.items():
            if not anchors:
                continue

            s_id = int(re.search(r'document(\d+)', s_path).group(1))
            doc_src = next((d for d in source_docs if d.numeric_id == s_id), None)
            if not doc_src:
                continue

            MAX_ANCHORS_PER_PAIR = 300
            if len(anchors) > MAX_ANCHORS_PER_PAIR:
                doc_len = len(doc_susp.text)
                bucket_size = max(1, doc_len // MAX_ANCHORS_PER_PAIR)
                buckets = {}
                for p_susp, p_src in anchors:
                    bucket = p_susp // bucket_size
                    if bucket not in buckets:
                        buckets[bucket] = (p_susp, p_src)
                anchors = list(buckets.values())

            for p_susp, p_src in anchors:
                if p_susp >= len(doc_susp.text) or p_src >= len(doc_src.text):
                    continue

                f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
                f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

                if not f_susp or not f_src:
                    continue

                t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
                t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

                if len(t_susp) > 30:
                    all_texts_susp.append(t_susp)
                    all_texts_src.append(t_src)
                    all_meta_pre_tfidf.append({
                        'susp_id': doc_susp.numeric_id,
                        'src_id':  s_id,
                        's_off':   s_off,
                        's_len':   s_len,
                        'src_off': src_off,
                        'src_len': src_len,
                    })

    t3_elapsed = time.time() - t3
    print(f"Candidate fragments: {len(all_texts_susp)} | Time: {t3_elapsed/60:.1f}min\n")

    filtered_texts_susp, filtered_texts_src, all_meta_post_tfidf = prefilter_with_tfidf(
        all_texts_susp, all_texts_src, all_meta_pre_tfidf
    )

    print(f"[STEP 5] SBERT batch encoding for {len(filtered_texts_susp)} pairs...")
    t5 = time.time()

    emb_susp = analyzer.model.encode(filtered_texts_susp, convert_to_tensor=True,
                                     show_progress_bar=True, batch_size=512)
    emb_src  = analyzer.model.encode(filtered_texts_src,  convert_to_tensor=True,
                                     show_progress_bar=True, batch_size=512)

    scores = torch.nn.functional.cosine_similarity(emb_susp, emb_src)

    print(f"Encoding done in {time.time() - t5:.1f}s\n")

    print("[STEP 6] Filtering and clustering with DBSCAN...")

    hits_per_pair = defaultdict(list)
    for i, score in enumerate(scores):
        if score >= threshold:
            m = all_meta_post_tfidf[i]
            hits_per_pair[(m['susp_id'], m['src_id'])].append(
                PredictedSegment(m['susp_id'], m['src_id'],
                                 m['s_off'], m['s_len'],
                                 m['src_off'], m['src_len'],
                                 score.item())
            )

    detections = defaultdict(list)
    for (susp_id, src_id), hits in hits_per_pair.items():
        clustered = cluster_results_dbscan(hits, eps=eps)
        detections[susp_id].extend(clustered)

    print(f"Pairs with detected plagiarism: {len(hits_per_pair)}\n")

    print("[STEP 7] Generating XML output...")
    os.makedirs("/content/output", exist_ok=True)
    for doc_susp in suspicious_docs:
        susp_id = doc_susp.numeric_id
        root = ET.Element("document", reference=doc_susp.doc_name)

        for det in detections.get(susp_id, []):
            ET.SubElement(root, "feature",
                name="detected-plagiarism",
                this_offset=str(det.susp_off),
                this_length=str(det.susp_len),
                source_reference=f"source-document{det.src_id:05d}.txt",
                source_offset=str(det.src_off),
                source_length=str(det.src_len)
            )

        tree = ET.ElementTree(root)
        out_path = f"/content/output/{doc_susp.doc_name.replace('.txt', '.xml')}"
        tree.write(out_path, encoding='utf-8', xml_declaration=True)

    print("[STEP 8] Evaluation vs Ground Truth...")
    total_tp = total_fp = total_fn = 0

    for doc_susp in suspicious_docs:
        susp_id = doc_susp.numeric_id
        gt_fragments = [(pf.this_offset, pf.this_offset + pf.this_length) for pf in doc_susp.plagiarism_features]
        det_fragments = [(d.susp_off, d.susp_off + d.susp_len) for d in detections.get(susp_id, [])]

        def get_covered(frags):
            s = set()
            for start, end in frags: s.update(range(start, end))
            return s

        gt_chars = get_covered(gt_fragments)
        det_chars = get_covered(det_fragments)

        tp = len(gt_chars & det_chars)
        fp = len(det_chars - gt_chars)
        fn = len(gt_chars - det_chars)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    p_g = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    r_g = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1_g = 2 * p_g * r_g / (p_g + r_g) if (p_g + r_g) > 0 else 0

    print("-" * 60)
    print(f"GLOBAL SCORES - Precision: {p_g:.4f}, Recall: {r_g:.4f}, F1: {f1_g:.4f}")
    print("-" * 60)

    print("\n" + "="*40)
    print("DETAILED TRACING OF GROUND TRUTH SEGMENTS")
    print("="*40)
    for doc_susp in suspicious_docs:
        debug_plagiarism_flow(doc_susp, detections, all_meta_pre_tfidf, all_meta_post_tfidf, scores_sbert=scores, threshold=threshold)

    return suspicious_docs, source_docs, detections

In [67]:
def debug_plagiarism_flow(doc_susp, detections, all_meta_pre_tfidf, all_meta_post_tfidf, scores_sbert=None, threshold=0.60):
    print(f"\n--- DETAILED ANALYSIS: {doc_susp.doc_name} ---")

    my_detections = detections.get(doc_susp.numeric_id, [])

    # --- PART A: GROUND TRUTH FRAGMENT VERIFICATION ---
    if doc_susp.plagiarism_features:
        print(f"Ground Truth properties: {len(doc_susp.plagiarism_features)} plagiarized fragments.")
        for pf in doc_susp.plagiarism_features:
            gt_start = pf.this_offset
            gt_end = pf.this_offset + pf.this_length
            src_target = pf.get_source_id()

            # 1. Encoplot Raw Anchors
            enc_hits = [m for m in all_meta_pre_tfidf if m['susp_id'] == doc_susp.numeric_id
                        and m['src_id'] == src_target
                        and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]

            # 2. TF-IDF Survival
            tfidf_hits = [m for m in all_meta_post_tfidf if m['susp_id'] == doc_susp.numeric_id
                          and m['src_id'] == src_target
                          and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]

            # 3. SBERT Score
            best_score = 0.0
            if scores_sbert is not None:
                relevant_indices = [i for i, m in enumerate(all_meta_post_tfidf)
                                   if m['susp_id'] == doc_susp.numeric_id
                                   and m['src_id'] == src_target
                                   and (m['s_off'] >= gt_start - 200 and m['s_off'] <= gt_end + 200)]
                if relevant_indices:
                    best_score = max([scores_sbert[i].item() for i in relevant_indices])

            # 4. Final Result Verification
            final_det = [d for d in my_detections if d.src_id == src_target
                         and (d.susp_off >= gt_start - 500 and d.susp_off <= gt_end + 500)]

            status = "✅ DETECTED" if final_det else "❌ MISSED"
            print(f"\nGT Fragment [{gt_start}:{gt_end}] (Source {src_target:05d}) -> {status}")
            print(f"   - Raw anchors (Encoplot): {len(enc_hits)}")
            print(f"   - After TF-IDF filter:    {len(tfidf_hits)}")
            if len(tfidf_hits) > 0:
                print(f"   - Best SBERT score:       {best_score:.4f}")

            if not final_det:
                if not enc_hits:
                    print("   [CAUSE]: Lexical Failure - Encoplot found no anchors.")
                elif not tfidf_hits:
                    print("   [CAUSE]: Statistical Failure - TF-IDF removed the fragment.")
                else:
                    print(f"   [CAUSE]: Semantic Failure - Score {best_score:.4f} < {threshold} or DBSCAN removed it.")
    else:
        print("Original document (according to Ground Truth).")

    # --- PART B: FALSE POSITIVE DETECTION (ERRONEOUSLY REPORTED FRAGMENTS) ---
    for det in my_detections:
        is_real = False
        for pf in doc_susp.plagiarism_features:
            # Check if our detection overlaps with any real fragment
            if not (det.susp_off + det.susp_len < pf.this_offset or pf.this_offset + pf.this_length < det.susp_off):
                is_real = True
                break

        if not is_real:
            print(f"\n FALSE POSITIVE DETECTED (Erroneously reported as plagiarism):")
            print(f"   - Position: [{det.susp_off}:{det.susp_off + det.susp_len}] | Source: {det.src_id:05d}")
            print(f"   - SBERT score of the error: {det.score:.4f}")

In [ ]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=10,
    analyzer=analyzer,
    threshold=0.65
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 10 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 10
Unique sources : 3
Necessary sources: 3

[STEP 2] Loading source docs...
Sources loaded: 3

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 3371 | Time: 0.1min

Prefilltering with tf-idf on 3371 pairs
  after tf-idf: 3095 remaining pairs (crossed out: 276)
[STEP 5] SBERT batch encoding for 3095 pairs...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding done in 45.7s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 7

[STEP 7] Generating XML output...
[STEP 8] Evaluation vs Ground Truth...
------------------------------------------------------------
GLOBAL SCORES - Precision: 0.9826, Recall: 0.8431, F1: 0.9075
------------------------------------------------------------

DETAILED TRACING OF GROUND TRUTH SEGMENTS

--- DETAILED ANALYSIS: suspicious-document00001.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00002.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00003.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00004.txt ---
Original document (according to Ground Truth).

--- DETAILED ANALYSIS: suspicious-document00005.txt ---
Ground Truth properties: 1 plagiarized fragments.

GT Fragment [19254:20811] (Source 00178) -> ✅ DETECTED
   - Raw anch

In [68]:
if 'analyzer' not in locals():
    analyzer = SemanticAnalyzer()

susp_docs, src_docs, final_detections = run_pipeline(
    n_suspicious=20,
    analyzer=analyzer,
    threshold=0.60
)

--------------------------------------------------------------------------------
[STEP 1] Extracting sample: 20 suspicious files
--------------------------------------------------------------------------------
Loaded suspicious docs : 20
Unique sources : 15
Necessary sources: 15

[STEP 2] Loading source docs...
Sources loaded: 15

[STEP 3] Encoplot - lexical radar...
Candidate fragments: 28001 | Time: 0.2min

Prefilltering with tf-idf on 28001 pairs
  after tf-idf: 24991 remaining pairs (crossed out: 3010)
[STEP 5] SBERT batch encoding for 24991 pairs...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Encoding done in 328.8s

[STEP 6] Filtering and clustering with DBSCAN...
Pairs with detected plagiarism: 75

[STEP 7] Generating XML output...
[STEP 8] Evaluation vs Ground Truth...
------------------------------------------------------------
GLOBAL SCORES - Precision: 0.8701, Recall: 0.6395, F1: 0.7372
------------------------------------------------------------

DETAILED TRACING OF GROUND TRUTH SEGMENTS

--- DETAILED ANALYSIS: suspicious-document00001.txt ---
Original document (according to Ground Truth).

 FALSE POSITIVE DETECTED (Erroneously reported as plagiarism):
   - Position: [23286:23736] | Source: 00374
   - SBERT score of the error: 0.6025

--- DETAILED ANALYSIS: suspicious-document00002.txt ---
Original document (according to Ground Truth).

 FALSE POSITIVE DETECTED (Erroneously reported as plagiarism):
   - Position: [30734:31195] | Source: 00374
   - SBERT score of the error: 0.6326

 FALSE POSITIVE DETECTED (Erroneously reported as plagiarism):
   - Position: [15273:15

In [2]:
import nbformat

with open('/content/drive/MyDrive/Colab Notebooks/licenta.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

for cell in nb.cells:
    if 'metadata' in cell:
        if 'executionInfo' in cell.metadata:
            del cell.metadata['executionInfo']

with open('/content/drive/MyDrive/Colab Notebooks/licenta_clean.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Done!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/licenta.ipynb'